In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the collected Bitcoin news dataset
news_path = Path("../data/news/raw/bitcoin_news_72h.csv")
news_df = pd.read_csv(news_path)

print("========== Dataset Overview ==========")
print("File:", news_path.resolve())
print("Total articles:", len(news_df))
print("Columns:", news_df.columns.tolist())

print("\n========== First 5 Articles ==========")
display(news_df.head())

========== Dataset Overview ==========
File: C:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\data\news\raw\bitcoin_news_72h.csv
Total articles: 122
Columns: ['title', 'source', 'published_at_utc', 'link', 'summary', 'content', 'content_status']

========== First 5 Articles ==========


,title,source,published_at_utc,link,summary,content,content_status
0,Bitcoin ETF Inflows Stall After $1.7 Billion B...,BeInCrypto,2026-10-12T04:09:36+00:00,https://finance.yahoo.com/markets/crypto/artic...,Bitcoin ETF inflows have totaled just $232 mil...,US spot Bitcoin (BTC) exchange-traded funds (E...,summary_and_content
1,Solana ETFs End a 14-Week Buying Streak With T...,BeInCrypto,2026-10-12T04:35:19+00:00,https://finance.yahoo.com/markets/crypto/artic...,"Solana ETFs lost $24.8 million last week, endi...",US spot Solana (SOL) exchange-traded funds (ET...,summary_and_content
2,IREN's Mystery Customer Is Anthropic? Fresh Bu...,Stocktwits,2026-10-12T01:57:34+00:00,https://finance.yahoo.com/technology/ai/articl...,A potential Anthropic partnership fuels fresh ...,Anthropic reportedly has an unannounced multi-...,summary_and_content
3,Eric Trump Says Crypto Should 'Embrace The Hel...,Benzinga,2026-10-11T21:30:06+00:00,https://finance.yahoo.com/markets/crypto/artic...,Eric Trump says artificial intelligence could ...,Benzinga and Yahoo Finance LLC may earn commis...,summary_and_content
4,Treasury Yields Are Now Above 5%. Here's Why T...,Motley Fool,2026-10-11T21:25:00+00:00,https://finance.yahoo.com/markets/crypto/artic...,Investors shouldn't conflate the devaluation o...,"The 10-Year Treasury yield recently hit 5.3%, ...",summary_and_content


In [3]:
# Check for duplicate records and missing values

print("========== 1. Duplicate Check ==========")
print("Duplicate links:", news_df["link"].duplicated().sum())
print("Duplicate titles:", news_df["title"].duplicated().sum())

print("\n========== 2. Missing Values ==========")
print(news_df[
    ["title", "source", "published_at_utc",
     "link", "summary", "content", "content_status"]
].isna().sum())

print("\n========== 3. Empty Text Check ==========")
print("Empty titles:",
      news_df["title"].fillna("").str.strip().eq("").sum())
print("Empty links:",
      news_df["link"].fillna("").str.strip().eq("").sum())
print("Empty content:",
      news_df["content"].fillna("").str.strip().eq("").sum())

print("\n========== 4. Content Status ==========")
print(news_df["content_status"].value_counts(dropna=False))

print("\n========== 5. Content Length ==========")
content_length = news_df["content"].fillna("").str.len()
print("Minimum content length:", content_length.min())
print("Content under 500 characters:", (content_length < 500).sum())

print("\n========== 6. Summary Check ==========")
summary = news_df["summary"].fillna("").str.strip()
content = news_df["content"].fillna("").str.strip()

print("Summary is None/empty:", summary.eq("").sum())
print("Summary identical to content:", (summary == content).sum())

========== 1. Duplicate Check ==========
Duplicate links: 0
Duplicate titles: 0

========== 2. Missing Values ==========
title               0
source              0
published_at_utc    0
link                0
summary             7
content             0
content_status      0
dtype: int64

========== 3. Empty Text Check ==========
Empty titles: 0
Empty links: 0
Empty content: 0

========== 4. Content Status ==========
content_status
summary_and_content    115
content_only             7
Name: count, dtype: int64

========== 5. Content Length ==========
Minimum content length: 99
Content under 500 characters: 7

========== 6. Summary Check ==========
Summary is None/empty: 7
Summary identical to content: 0


In [4]:
# Check publication timestamps against the latest 72-hour window

from datetime import datetime, timezone

check_time = datetime.now(timezone.utc)
cutoff_time = pd.Timestamp(check_time) - pd.Timedelta(hours=72)

news_df["published_at_utc"] = pd.to_datetime(
    news_df["published_at_utc"],
    utc=True,
    errors="coerce"
)

invalid_timestamps = news_df["published_at_utc"].isna()

outside_window = (
    (news_df["published_at_utc"] < cutoff_time) |
    (news_df["published_at_utc"] > pd.Timestamp(check_time))
) & ~invalid_timestamps

print("========== Publication Time Check ==========")
print("Check time (UTC):", check_time.isoformat())
print("72-hour cutoff (UTC):", cutoff_time.isoformat())
print("Invalid timestamps:", invalid_timestamps.sum())
print("Articles outside 72-hour window:", outside_window.sum())

valid_times = news_df.loc[
    ~invalid_timestamps, "published_at_utc"
]

if not valid_times.empty:
    print("Newest article (UTC):", valid_times.max())
    print("Oldest article (UTC):", valid_times.min())

if outside_window.any():
    print("\nArticles outside the 72-hour window:")
    display(
        news_df.loc[
            outside_window,
            ["title", "published_at_utc", "link"]
        ]
    )

# Filter articles within the latest 72 hours
recent_news_df = news_df.loc[
    news_df["published_at_utc"].between(
        cutoff_time,
        pd.Timestamp(check_time),
        inclusive="both"
    )
].copy()

print("Articles within the latest 72 hours:", len(recent_news_df))

========== Publication Time Check ==========
Check time (UTC): 2026-10-12T06:20:24.805038+00:00
72-hour cutoff (UTC): 2026-10-09T06:20:24.805038+00:00
Invalid timestamps: 0
Articles outside 72-hour window: 1
Newest article (UTC): 2026-10-12 04:35:19+00:00
Oldest article (UTC): 2026-10-09 06:11:53+00:00

Articles outside the 72-hour window:


,title,published_at_utc,link
121,Twenty One Capital (XXI) Rebound Puts Its Valu...,2026-10-09 06:11:53+00:00,https://finance.yahoo.com/markets/crypto/artic...


Articles within the latest 72 hours: 121


In [5]:
# Identify articles that may be unrelated to Bitcoin or cryptocurrency

import re

crypto_keywords = [
    "bitcoin", "btc", "cryptocurrency", "crypto",
    "blockchain", "digital asset", "digital assets",
    "bitcoin etf", "crypto etf"
]

def check_crypto_relevance(row):
    text = f"{row['title']} {row['summary']} {row['content']}"
    text = text.lower()
    return any(
        re.search(r"\b" + re.escape(keyword) + r"\b", text)
        for keyword in crypto_keywords
    )

news_df["keyword_relevant"] = news_df.apply(
    check_crypto_relevance, axis=1
)

potentially_irrelevant = news_df[
    ~news_df["keyword_relevant"]
]

print("Total articles:", len(news_df))
print("Potentially relevant:", news_df["keyword_relevant"].sum())
print("Potentially irrelevant:", len(potentially_irrelevant))

display(
    potentially_irrelevant[
        ["title", "source", "summary"]
    ]
)

Total articles: 122
Potentially relevant: 118
Potentially irrelevant: 4


,title,source,summary
65,Sector Update: Financial Stocks Rise Friday,MT Newswires,NaN
70,Sector Update: Financial Stocks Advance Friday...,MT Newswires,NaN
85,"Exchange-Traded Funds, Equity Futures Up Pre-B...",MT Newswires,NaN
94,Stock Market Today: Dow Pops As Indexes Finish...,Investor's Business Daily,NaN


In [6]:
# Inspect potentially irrelevant articles before making any decisions

review_columns = ["title", "source", "link", "content"]

for index, row in potentially_irrelevant.iterrows():
    print("\n" + "=" * 100)
    print(f"Index: {index}")
    print(f"Title: {row['title']}")
    print(f"Source: {row['source']}")
    print(f"Link: {row['link']}")
    print(f"Content length: {len(str(row['content']))}")
    print("\nFull content:")
    print(row["content"])


Index: 65
Title: Sector Update: Financial Stocks Rise Friday
Source: MT Newswires
Link: https://finance.yahoo.com/markets/stocks/articles/sector-financial-stocks-rise-friday-200414075.html
Content length: 100

Full content:
Financial stocks rose Friday with the NYSE Financial Index and the State Street Financial Select Sec

Index: 70
Title: Sector Update: Financial Stocks Advance Friday Afternoon
Source: MT Newswires
Link: https://finance.yahoo.com/markets/stocks/articles/sector-financial-stocks-advance-friday-175748005.html
Content length: 99

Full content:
Financial stocks were higher in Friday afternoon trading, with the NYSE Financial Index rising 0.6%

Index: 85
Title: Exchange-Traded Funds, Equity Futures Up Pre-Bell Friday as Oil Prices Fall, Traders Assess SpaceX Spectrum Deal
Source: MT Newswires
Link: https://finance.yahoo.com/markets/stocks/articles/exchange-traded-funds-equity-futures-132619334.html
Content length: 100

Full content:
The broad market exchange-traded fund S

In [7]:
# Keyword-based filtering after the 72-hour date filter
crypto_keywords = [
    "bitcoin",
    "btc",
    "cryptocurrency",
    "cryptocurrencies",
    "crypto",
    "blockchain",
    "digital asset",
    "digital assets"
]

def is_crypto_related(row):
    fields = [
        row.get("title", ""),
        row.get("summary", ""),
        row.get("content", "")
    ]

    text = " ".join(
        "" if pd.isna(value) else str(value)
        for value in fields
    ).lower()

    return any(
        re.search(
            r"\b" + re.escape(keyword) + r"\b",
            text
        )
        for keyword in crypto_keywords
    )


# Keep the original datasets unchanged
keyword_mask = recent_news_df.apply(
    is_crypto_related, axis=1
)

keyword_filtered_df = recent_news_df.loc[
    keyword_mask
].copy()

excluded_news_df = recent_news_df.loc[
    ~keyword_mask
].copy()

print("========== Keyword Filtering Summary ==========")
print("Articles after date filtering:", len(recent_news_df))
print("Keyword-relevant articles:", len(keyword_filtered_df))
print("Articles excluded by keyword filter:", len(excluded_news_df))

print("\n========== Excluded Articles ==========")
display(
    excluded_news_df[
        ["title", "source", "published_at_utc"]
    ]
)


========== Keyword Filtering Summary ==========
Articles after date filtering: 121
Keyword-relevant articles: 117
Articles excluded by keyword filter: 4

========== Excluded Articles ==========


,title,source,published_at_utc
65,Sector Update: Financial Stocks Rise Friday,MT Newswires,2026-10-09 20:04:14+00:00
70,Sector Update: Financial Stocks Advance Friday...,MT Newswires,2026-10-09 17:57:48+00:00
85,"Exchange-Traded Funds, Equity Futures Up Pre-B...",MT Newswires,2026-10-09 13:26:19+00:00
94,Stock Market Today: Dow Pops As Indexes Finish...,Investor's Business Daily,2026-10-09 20:28:44+00:00


In [8]:
print("Date-filtered articles:", len(recent_news_df))
print("Keyword-filtered articles:", len(keyword_filtered_df))
print("Excluded articles:", len(excluded_news_df))

print("\nRetained article sample:")
display(
    keyword_filtered_df[
        ["title", "source", "published_at_utc"]
    ].head(5)
)

Date-filtered articles: 121
Keyword-filtered articles: 117
Excluded articles: 4

Retained article sample:


,title,source,published_at_utc
0,Bitcoin ETF Inflows Stall After $1.7 Billion B...,BeInCrypto,2026-10-12 04:09:36+00:00
1,Solana ETFs End a 14-Week Buying Streak With T...,BeInCrypto,2026-10-12 04:35:19+00:00
2,IREN's Mystery Customer Is Anthropic? Fresh Bu...,Stocktwits,2026-10-12 01:57:34+00:00
3,Eric Trump Says Crypto Should 'Embrace The Hel...,Benzinga,2026-10-11 21:30:06+00:00
4,Treasury Yields Are Now Above 5%. Here's Why T...,Motley Fool,2026-10-11 21:25:00+00:00


## SBERT

In [9]:
# Load the SBERT model
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

print("SBERT model loaded successfully.")
print("Embedding dimension:", sbert_model.get_sentence_embedding_dimension())

INFO: No device provided, using cpu
INFO: Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5341.34it/s]


SBERT model loaded successfully.
Embedding dimension: 384


C:\Users\nyy32\AppData\Local\Temp\ipykernel_240436\4262035451.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", sbert_model.get_sentence_embedding_dimension())


In [10]:
# Define a neutral forecasting query
forecasting_query = """
Retrieve recent financial news relevant to Bitcoin (BTC) price forecasting,
including Bitcoin price movements, ETF flows, cryptocurrency market developments,
investor sentiment, regulatory developments, institutional adoption, and
macroeconomic events that may influence Bitcoin prices.
"""

# Prepare news text for SBERT
def build_article_text(row):
    text_parts = []

    for column in ["title", "summary", "content"]:
        value = row.get(column)

        if pd.notna(value) and str(value).strip():
            text_parts.append(
                f"{column.capitalize()}: {str(value).strip()}"
            )

    return "\n".join(text_parts)


news_texts = [
    build_article_text(row)
    for _, row in keyword_filtered_df.iterrows()
]

print("========== Query and Text Preparation ==========")
print("Forecasting query prepared:", bool(forecasting_query.strip()))
print("Candidate articles:", len(news_texts))
print("Empty article texts:", sum(not text.strip() for text in news_texts))

print("\n========== Forecasting Query ==========")
print(forecasting_query.strip())

print("\n========== First Article Text Preview ==========")
print(news_texts[0][:600] if news_texts else "No articles available.")

========== Query and Text Preparation ==========
Forecasting query prepared: True
Candidate articles: 117
Empty article texts: 0

========== Forecasting Query ==========
Retrieve recent financial news relevant to Bitcoin (BTC) price forecasting,
including Bitcoin price movements, ETF flows, cryptocurrency market developments,
investor sentiment, regulatory developments, institutional adoption, and
macroeconomic events that may influence Bitcoin prices.

========== First Article Text Preview ==========
Title: Bitcoin ETF Inflows Stall After $1.7 Billion Burst: Will Demand Return?
Summary: Bitcoin ETF inflows have totaled just $232 million in 13 sessions since a $1.7 billion burst. Can buyers return?
Content: US spot Bitcoin (BTC) exchange-traded funds (ETFs) have added only about $232 million over 13 sessions. Bitcoin ETF inflows totaled $1.7 billion across September 21 and 22, then went quiet.

Bitcoin's price recovered to above $83,000 overnight while US-listed funds were not trading,

In [11]:
# 1. Encode the forecasting query
query_embedding = sbert_model.encode(
    forecasting_query,
    convert_to_tensor=True,
    show_progress_bar=False
)

# 2. Encode all candidate news articles
news_embeddings = sbert_model.encode(
    news_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

# 3. Calculate cosine similarity
similarity_scores = util.cos_sim(
    query_embedding,
    news_embeddings
).cpu().numpy().flatten()

# 4. Create a separate results DataFrame
semantic_results_df = keyword_filtered_df.copy()

semantic_results_df["similarity_score"] = similarity_scores

# 5. Rank articles by semantic similarity
semantic_results_df = (
    semantic_results_df
    .sort_values(
        by="similarity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

semantic_results_df["similarity_rank"] = (
    range(1, len(semantic_results_df) + 1)
)

# 6. Display the retrieval summary
print("========== SBERT Retrieval Summary ==========")
print("Candidate articles:", len(news_texts))
print("Query embedding dimension:", query_embedding.shape[-1])
print("News embeddings shape:", tuple(news_embeddings.shape))
print("Similarity scores calculated:", len(similarity_scores))

print("\n========== Top 10 Retrieved Articles ==========")

display(
    semantic_results_df[
        [
            "similarity_rank",
            "title",
            "source",
            "published_at_utc",
            "similarity_score"
        ]
    ].head(10)
)

Batches: 100%|██████████| 4/4 [00:02<00:00,  1.87it/s]

========== SBERT Retrieval Summary ==========
Candidate articles: 117
Query embedding dimension: 384
News embeddings shape: (117, 384)
Similarity scores calculated: 117

========== Top 10 Retrieved Articles ==========


,similarity_rank,title,source,published_at_utc,similarity_score
0,1,Bitcoin Could Be Entering a New Bull Market De...,BeInCrypto,2026-10-11 19:22:47+00:00,0.626807
1,2,"Bitcoin and ethereum prices today, Friday, Oct...",Yahoo Personal Finance,2026-10-09 11:33:50+00:00,0.611319
2,3,"Bitcoin rebounds above $83,000 after weekly sl...",Investing.com,2026-10-10 04:29:57+00:00,0.591106
3,4,"Bitcoin Rebounds After Rough Week, But Traders...",decrypt,2026-10-09 17:36:26+00:00,0.588015
4,5,"Bitcoin, Strategy, and Crypto Stocks Rebound a...",Barrons.com,2026-10-09 12:21:00+00:00,0.583545
5,6,Bitcoin Tests Average Buyer’s Break-Even As ET...,Stocktwits,2026-10-10 17:34:25+00:00,0.576363
6,7,Weekly Wrap: Rising Treasury Yields Sink Bitcoin,CryptoProwl,2026-10-09 20:52:00+00:00,0.569976
7,8,"Bitcoin struggles above $83,000 a year after $...",Investing.com,2026-10-11 05:54:26+00:00,0.559304
8,9,AI Price Prediction: Insights from Grok on Bit...,24/7 Wall St.,2026-10-11 12:12:14+00:00,0.544206
9,10,"Bitcoin holds ground above $82k, heads for wee...",Investing.com,2026-10-09 07:05:55+00:00,0.537582


In [12]:
# Select the top 10 retrieved news articles

TOP_K = 10

retrieved_news_df = semantic_results_df.head(TOP_K).copy()

print("Retrieved news prepared successfully.")
print("Selected articles:", len(retrieved_news_df))
print(
    "Similarity score range:",
    round(retrieved_news_df["similarity_score"].min(), 4),
    "to",
    round(retrieved_news_df["similarity_score"].max(), 4)
)

Retrieved news prepared successfully.
Selected articles: 10
Similarity score range: 0.5376 to 0.6268


In [13]:
print("========== Retrieved News Quality Check ==========")

print("Retrieved articles:", len(retrieved_news_df))
print(
    "Duplicate links:",
    retrieved_news_df["link"].duplicated().sum()
)
print(
    "Missing titles:",
    retrieved_news_df["title"].isna().sum()
)
print(
    "Missing links:",
    retrieved_news_df["link"].isna().sum()
)
print(
    "Scores sorted descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)

========== Retrieved News Quality Check ==========
Retrieved articles: 10
Duplicate links: 0
Missing titles: 0
Missing links: 0
Scores sorted descending: True


In [14]:
# Review the relevance of the top 10 retrieved news articles

for _, row in retrieved_news_df.iterrows():
    print("\n" + "=" * 90)
    print("Rank:", row["similarity_rank"])
    print("Similarity score:", round(row["similarity_score"], 4))
    print("Title:", row["title"])
    print("Source:", row["source"])
    print("Published (UTC):", row["published_at_utc"])

    summary = row.get("summary")
    content = row.get("content")

    print("\nSummary:")
    print(summary if pd.notna(summary) else "None")

    print("\nContent preview:")
    print(str(content)[:600] if pd.notna(content) else "None")


Rank: 1
Similarity score: 0.6268
Title: Bitcoin Could Be Entering a New Bull Market Despite Latest Sell-Off
Source: BeInCrypto
Published (UTC): 2026-10-11 19:22:47+00:00

Summary:
BloFin Research says Bitcoin's new bull market remains intact despite the latest sell-off, as historical data suggests its boom-and-bust cycles are weakening.

Content preview:
Despite the market dip and liquidations over the past week, Bitcoin has seemingly entered a bull market, as the analysis of its 50-week moving average shows.

BloFin Research argues that Bitcoin's traditional four-year boom-and-bust cycle is gradually losing strength. Future rallies could deliver smaller gains, followed by shorter and less severe downturns.

The strongest evidence comes from Bitcoin's recovery above its 50-week moving average, a technical indicator that previously signaled bull-market transitions in 2015, 2019 and 2023.

Bitcoin reclaimed this level near $78,800 in September. 

Rank: 2
Similarity score: 0.6113
Title: 

In [15]:
# Remove identifiable promotional paragraphs from retrieval text only
ad_patterns = [
    r"Before you buy NVIDIA",
    r"stock picks are beating the market",
    r"See the list now",
    r"See what 10M\+ investors are talking about",
    r"Get the Stocktwits Daily Rip",
    r"Building a portfolio and living off one",
    r"\(Sponsor\)"
]

def build_clean_article_text(row):
    parts = []

    title = row.get("title")
    summary = row.get("summary")
    content = row.get("content")

    if pd.notna(title) and str(title).strip():
        parts.append(f"Title: {str(title).strip()}")

    if pd.notna(summary) and str(summary).strip():
        parts.append(f"Summary: {str(summary).strip()}")

    if pd.notna(content) and str(content).strip():
        paragraphs = re.split(
            r"\n\s*\n",
            str(content).strip()
        )

        clean_paragraphs = [
            paragraph.strip()
            for paragraph in paragraphs
            if paragraph.strip()
            and not any(
                re.search(pattern, paragraph, flags=re.IGNORECASE)
                for pattern in ad_patterns
            )
        ]

        if clean_paragraphs:
            parts.append(
                "Content: " + "\n\n".join(clean_paragraphs)
            )

    return "\n".join(parts)


# Create clean retrieval texts without changing the source DataFrame
news_texts_clean = [
    build_clean_article_text(row)
    for _, row in keyword_filtered_df.iterrows()
]

print("========== Retrieval Text Cleaning ==========")
print("Candidate articles:", len(news_texts_clean))
print("Empty article texts:", sum(
    not text.strip() for text in news_texts_clean
))

print("\n========== Cleaned Example ==========")
print(news_texts_clean[2][:1000])

========== Retrieval Text Cleaning ==========
Candidate articles: 117
Empty article texts: 0

========== Cleaned Example ==========
Title: IREN's Mystery Customer Is Anthropic? Fresh Buzz Pushes Stock Higher
Summary: A potential Anthropic partnership fuels fresh optimism for the Bitcoin miner-turned-AI infrastructure provider, even as operational concerns linger.
Content: Anthropic reportedly has an unannounced multi-year cloud deal with IREN, though neither company has confirmed the partnership.

The potential deal could strengthen IREN's position in the booming AI infrastructure market.

IREN trends in Stocktwits' trending list as retail traders turn 'bullish.'

IREN Ltd.'s (IREN) shares jumped 8% in overnight trading late Sunday after a report said that Anthropic is the mystery AI cloud customer the company alluded to during its earnings call a few months ago.

The Information published an analysis of the economics of Anthropic's data center leases, detailing how the AI lab is pushi

In [16]:
# 1. Encode the same forecasting query
query_embedding_clean = sbert_model.encode(
    forecasting_query,
    convert_to_tensor=True,
    show_progress_bar=False
)

# 2. Re-encode the cleaned news texts
news_embeddings_clean = sbert_model.encode(
    news_texts_clean,
    convert_to_tensor=True,
    show_progress_bar=True
)

# 3. Recalculate cosine similarity
clean_scores = util.cos_sim(
    query_embedding_clean,
    news_embeddings_clean
).cpu().numpy().flatten()

# 4. Create a new results DataFrame
cleaned_semantic_results_df = keyword_filtered_df.copy()
cleaned_semantic_results_df["similarity_score"] = clean_scores

cleaned_semantic_results_df = (
    cleaned_semantic_results_df
    .sort_values("similarity_score", ascending=False)
    .reset_index(drop=True)
)

cleaned_semantic_results_df["similarity_rank"] = range(
    1, len(cleaned_semantic_results_df) + 1
)

# 5. Compare old and new rankings
rank_comparison = semantic_results_df[
    ["link", "title", "similarity_rank", "similarity_score"]
].merge(
    cleaned_semantic_results_df[
        ["link", "similarity_rank", "similarity_score"]
    ],
    on="link",
    suffixes=("_before", "_after"),
    validate="one_to_one"
)

# 6. Display the new Top 10
print("========== Cleaned SBERT Retrieval Summary ==========")
print("Candidate articles:", len(cleaned_semantic_results_df))
print("New embeddings shape:", tuple(news_embeddings_clean.shape))
print("Similarity scores recalculated:", len(clean_scores))

print("\n========== Top 10 After Cleaning ==========")
display(
    cleaned_semantic_results_df[
        [
            "similarity_rank",
            "title",
            "source",
            "similarity_score"
        ]
    ].head(10)
)

print("\n========== Ranking Comparison ==========")
old_top10_links = set(
    semantic_results_df.head(10)["link"]
)
new_top10_links = set(
    cleaned_semantic_results_df.head(10)["link"]
)

print("Top 10 articles retained:", len(old_top10_links & new_top10_links))
print("Articles entering Top 10:", len(new_top10_links - old_top10_links))
print("Articles leaving Top 10:", len(old_top10_links - new_top10_links))

Batches: 100%|██████████| 4/4 [00:02<00:00,  1.92it/s]

========== Cleaned SBERT Retrieval Summary ==========
Candidate articles: 117
New embeddings shape: (117, 384)
Similarity scores recalculated: 117

========== Top 10 After Cleaning ==========


,similarity_rank,title,source,similarity_score
0,1,Bitcoin Could Be Entering a New Bull Market De...,BeInCrypto,0.626807
1,2,AI Price Prediction: Insights from Grok on Bit...,24/7 Wall St.,0.613354
2,3,"Bitcoin and ethereum prices today, Friday, Oct...",Yahoo Personal Finance,0.611319
3,4,"Bitcoin rebounds above $83,000 after weekly sl...",Investing.com,0.588803
4,5,"Bitcoin Rebounds After Rough Week, But Traders...",decrypt,0.588015
5,6,"Bitcoin struggles above $83,000 a year after $...",Investing.com,0.587221
6,7,"Bitcoin, Strategy, and Crypto Stocks Rebound a...",Barrons.com,0.583545
7,8,Bitcoin Tests Average Buyer’s Break-Even As ET...,Stocktwits,0.576397
8,9,Weekly Wrap: Rising Treasury Yields Sink Bitcoin,CryptoProwl,0.569976
9,10,"Bitcoin holds ground above $82k, heads for wee...",Investing.com,0.568776



========== Ranking Comparison ==========
Top 10 articles retained: 10
Articles entering Top 10: 0
Articles leaving Top 10: 0


In [17]:
# Update the final retrieved news using cleaned SBERT results

TOP_K = 10

retrieved_news_df = (
    cleaned_semantic_results_df
    .head(TOP_K)
    .copy()
)

print("========== Final Retrieved News ==========")
print("Candidate articles:", len(cleaned_semantic_results_df))
print("Retrieved articles:", len(retrieved_news_df))
print(
    "Similarity scores descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)

display(
    retrieved_news_df[
        [
            "similarity_rank",
            "title",
            "source",
            "published_at_utc",
            "similarity_score"
        ]
    ]
)

========== Final Retrieved News ==========
Candidate articles: 117
Retrieved articles: 10
Similarity scores descending: True


,similarity_rank,title,source,published_at_utc,similarity_score
0,1,Bitcoin Could Be Entering a New Bull Market De...,BeInCrypto,2026-10-11 19:22:47+00:00,0.626807
1,2,AI Price Prediction: Insights from Grok on Bit...,24/7 Wall St.,2026-10-11 12:12:14+00:00,0.613354
2,3,"Bitcoin and ethereum prices today, Friday, Oct...",Yahoo Personal Finance,2026-10-09 11:33:50+00:00,0.611319
3,4,"Bitcoin rebounds above $83,000 after weekly sl...",Investing.com,2026-10-10 04:29:57+00:00,0.588803
4,5,"Bitcoin Rebounds After Rough Week, But Traders...",decrypt,2026-10-09 17:36:26+00:00,0.588015
5,6,"Bitcoin struggles above $83,000 a year after $...",Investing.com,2026-10-11 05:54:26+00:00,0.587221
6,7,"Bitcoin, Strategy, and Crypto Stocks Rebound a...",Barrons.com,2026-10-09 12:21:00+00:00,0.583545
7,8,Bitcoin Tests Average Buyer’s Break-Even As ET...,Stocktwits,2026-10-10 17:34:25+00:00,0.576397
8,9,Weekly Wrap: Rising Treasury Yields Sink Bitcoin,CryptoProwl,2026-10-09 20:52:00+00:00,0.569976
9,10,"Bitcoin holds ground above $82k, heads for wee...",Investing.com,2026-10-09 07:05:55+00:00,0.568776


In [18]:
print("========== Final SBERT Retrieval Quality Check ==========")

print("Candidate articles:", len(cleaned_semantic_results_df))
print("Retrieved articles:", len(retrieved_news_df))

print(
    "Duplicate links:",
    retrieved_news_df["link"].duplicated().sum()
)

print(
    "Missing titles:",
    retrieved_news_df["title"].isna().sum()
)

print(
    "Missing links:",
    retrieved_news_df["link"].isna().sum()
)

print(
    "Missing content:",
    retrieved_news_df["content"].isna().sum()
)

print(
    "Missing similarity scores:",
    retrieved_news_df["similarity_score"].isna().sum()
)

print(
    "Scores sorted descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)

print(
    "All scores finite:",
    np.isfinite(retrieved_news_df["similarity_score"]).all()
)

print("\nSimilarity score range:")
print(
    round(retrieved_news_df["similarity_score"].min(), 4),
    "to",
    round(retrieved_news_df["similarity_score"].max(), 4)
)

========== Final SBERT Retrieval Quality Check ==========
Candidate articles: 117
Retrieved articles: 10
Duplicate links: 0
Missing titles: 0
Missing links: 0
Missing content: 0
Missing similarity scores: 0
Scores sorted descending: True
All scores finite: True

Similarity score range:
0.5688 to 0.6268


In [19]:
# Map cleaned article texts back to their original article links
clean_text_by_link = dict(
    zip(keyword_filtered_df["link"], news_texts_clean)
)

cleaned_semantic_results_df["retrieval_text"] = (
    cleaned_semantic_results_df["link"].map(clean_text_by_link)
)

# Refresh the final Top 10 using cleaned SBERT rankings
TOP_K = 10

retrieved_news_df = (
    cleaned_semantic_results_df
    .head(TOP_K)
    .copy()
)

# Select the fields needed for downstream interpretation
columns_to_save = [
    "similarity_rank",
    "similarity_score",
    "title",
    "source",
    "published_at_utc",
    "link",
    "summary",
    "content",
    "retrieval_text",
    "content_status"
]

# Save to the project-level processed news directory
output_dir = Path("../data/news/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "bitcoin_news_retrieved_top10.csv"

retrieved_news_df[columns_to_save].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Retrieved news saved successfully.")
print("File:", output_path)
print("Saved articles:", len(retrieved_news_df))
print(
    "Duplicate links:",
    retrieved_news_df["link"].duplicated().sum()
)
print(
    "Scores sorted descending:",
    retrieved_news_df["similarity_score"].is_monotonic_decreasing
)
print("Columns:", columns_to_save)

Retrieved news saved successfully.
File: ..\data\news\processed\bitcoin_news_retrieved_top10.csv
Saved articles: 10
Duplicate links: 0
Scores sorted descending: True
Columns: ['similarity_rank', 'similarity_score', 'title', 'source', 'published_at_utc', 'link', 'summary', 'content', 'retrieval_text', 'content_status']
